# Maximum Likelihood Estimation

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 03.06 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/07_maximum_likelihood_estimation.ipynb)

---

## 🎯 Learning Objective

Derive maximum likelihood estimation from first principles and connect it directly to the cross-entropy loss used to train neural networks.

---

## 📐 Theory

Every time you train a model by minimizing a loss function, you are almost certainly doing
maximum likelihood estimation, whether or not you've framed it that way. This notebook makes
that connection explicit and precise.

### The likelihood function

Given data $D=\{x_1,\ldots,x_n\}$ assumed drawn i.i.d. from a distribution with unknown
parameter $\theta$, the **likelihood** is the joint probability of the data, viewed as a
function of $\theta$ with the data held fixed:

$$L(\theta) = P(D\mid\theta) = \prod_{i=1}^n p(x_i\mid\theta),$$

the product coming directly from independence (`03.04`). This is the *same* object that
appears in the numerator of Bayes' theorem (`03.05`) — MLE is exactly MAP estimation with a
flat (uninformative) prior, so whatever $\theta$ makes the observed data most probable is
declared the estimate:

$$\hat\theta_{\text{MLE}} = \arg\max_\theta L(\theta) = \arg\max_\theta \prod_{i=1}^n p(x_i\mid\theta).$$

### Log-likelihood: why we always take the log

Products of many small probabilities underflow numerically and are painful to differentiate.
The logarithm is strictly increasing, so it never changes *where* the maximum is, while turning
the product into a sum:

$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^n \log p(x_i\mid\theta), \qquad \hat\theta_{\text{MLE}} = \arg\max_\theta \ell(\theta).$$

Setting $\nabla_\theta \ell(\theta) = 0$ and solving (or, when there's no closed form, running
gradient *ascent* on $\ell$, or equivalently gradient *descent* on $-\ell$) is how MLE is
actually computed in practice — precisely the mechanics of `02.05`
(link: [Gradients and Directional Derivatives](../02_Calculus/05_gradients_and_directional_derivatives.ipynb)).

### Cross-entropy loss IS negative log-likelihood

For a classifier producing $\hat p_i = P(\text{class}=y_i\mid x_i;\theta)$ (a softmax output,
`03.02`) on a labeled dataset, the log-likelihood is $\ell(\theta) = \sum_i \log \hat p_i$. The
**cross-entropy loss** used to train the network is defined as
$\mathcal{L}_{\text{CE}}(\theta) = -\frac{1}{n}\sum_i \log \hat p_i$ — literally $-\ell(\theta)/n$.
Therefore:

$$\arg\min_\theta \mathcal{L}_{\text{CE}}(\theta) = \arg\min_\theta \left(-\frac{1}{n}\ell(\theta)\right) = \arg\max_\theta \ell(\theta) = \hat\theta_{\text{MLE}}.$$

Minimizing cross-entropy and maximizing likelihood are the *same optimization problem*, up to
the sign flip and the constant $1/n$ that don't change where the optimum is. This equivalence
is not an analogy — it's an algebraic identity, demonstrated numerically below.

### Fisher information and asymptotic properties

The **Fisher information** $I(\theta) = \mathbb{E}\left[\left(\frac{\partial}{\partial\theta}
\log p(X\mid\theta)\right)^2\right] = -\mathbb{E}\left[\frac{\partial^2}{\partial\theta^2}\log
p(X\mid\theta)\right]$ measures how sharply the log-likelihood curves at the true $\theta$: high
Fisher information means a sharp peak, so data is highly informative about $\theta$. Under
standard regularity conditions, the MLE is **consistent** ($\hat\theta_{\text{MLE}} \to
\theta_{\text{true}}$ as $n\to\infty$) and **asymptotically normal**,
$\hat\theta_{\text{MLE}} \approx \mathcal{N}(\theta_{\text{true}}, 1/(nI(\theta)))$ — variance
shrinks as $1/n$, scaled inversely by how informative each observation is. This asymptotic
variance $1/(nI(\theta))$ is the **Cramér–Rao lower bound** — the smallest variance any
unbiased estimator can achieve (Cramér 1946; Rao 1945; proof in Casella & Berger, Ch. 7,
cited in Further Reading) — so the MLE is *asymptotically efficient*: no unbiased estimator
does better in the large-$n$ limit.

### The EM algorithm

When some data is unobserved — a **latent variable** $z_i$ per data point, as in a mixture
model — the log-likelihood $\log\sum_z p(x,z\mid\theta)$ involves a log of a sum, which usually
has no closed-form maximum. The **Expectation-Maximization (EM) algorithm** breaks this into
two tractable steps, iterated to convergence: the **E-step** computes the posterior
responsibility $P(z_i\mid x_i,\theta^{(t)})$ of each latent value under the current parameters
(Bayes' theorem again), and the **M-step** maximizes the *expected complete-data
log-likelihood* under those responsibilities — now a log of individual terms, so it usually has
a closed form. Each EM iteration provably never decreases the observed log-likelihood, and it's
the backbone of fitting the Gaussian Mixture Models in
[Probabilistic Models in Machine Learning](10_probabilistic_models_in_ml.ipynb) (`03.10`).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the log-likelihood of a Gaussian's mean $\mu$ against candidate values, side by side
with the negative log-likelihood, makes "maximizing likelihood = minimizing negative
log-likelihood" visually obvious: one curve is a vertical flip of the other, and they peak/dip
at the exact same $\mu$.

In [ ]:
# Log-likelihood vs. negative log-likelihood for a Gaussian's mean mu, with sigma known/fixed --
# one is the exact vertical flip of the other, so their optimizers coincide.
rng = np.random.default_rng(0)
true_mu, sigma = 3.0, 1.5
data = rng.normal(true_mu, sigma, size=60)

mu_grid = np.linspace(0, 6, 300)
log_lik = np.array([np.sum(norm.logpdf(data, loc=mu, scale=sigma)) for mu in mu_grid])
neg_log_lik = -log_lik

mle_mu_numeric = mu_grid[np.argmax(log_lik)]
mle_mu_closed_form = data.mean()   # for a Gaussian with known sigma, MLE of mu IS the sample mean

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(mu_grid, log_lik, color="#4C72B0", lw=2)
axes[0].axvline(mle_mu_closed_form, color="red", ls="--", label=f"argmax = {mle_mu_closed_form:.3f}")
axes[0].set_title("Log-likelihood ℓ(μ)  [maximize this]")
axes[0].set_xlabel("μ"); axes[0].set_ylabel("ℓ(μ)"); axes[0].legend()

axes[1].plot(mu_grid, neg_log_lik, color="#C44E52", lw=2)
axes[1].axvline(mle_mu_closed_form, color="red", ls="--", label=f"argmin = {mle_mu_closed_form:.3f}")
axes[1].set_title("Negative log-likelihood -ℓ(μ)  [minimize this]")
axes[1].set_xlabel("μ"); axes[1].set_ylabel("-ℓ(μ)"); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"argmax of log-likelihood (grid search):      mu = {mle_mu_numeric:.4f}")
print(f"argmin of negative log-likelihood (grid search): mu = {mu_grid[np.argmin(neg_log_lik)]:.4f}")
print(f"Closed-form MLE (sample mean):                 mu = {mle_mu_closed_form:.4f}")
print(f"True mu used to generate the data:              mu = {true_mu}")
print("All three agree (up to grid resolution) -- maximizing L and minimizing -L pick the same point.")

## 🐍 Implementation from Scratch

We derive and implement the closed-form MLE for a Gaussian's mean and variance, verify the
Fisher-information-predicted sampling variance of the MLE via simulation, and implement the EM
algorithm from scratch for a 2-component Gaussian mixture with a latent cluster assignment.

In [ ]:
# MLE of a Gaussian's mean/variance, in closed form (found by setting d(log-lik)/d(param)=0)
def gaussian_mle(data):
    mu_hat = data.mean()                        # argmax_mu of the log-likelihood: the sample mean
    sigma2_hat = np.mean((data - mu_hat) ** 2)   # argmax_sigma2: the (biased) sample variance
    return mu_hat, sigma2_hat

rng = np.random.default_rng(1)
true_mu, true_sigma = 3.0, 2.0
sample = rng.normal(true_mu, true_sigma, size=500)
mu_hat, sigma2_hat = gaussian_mle(sample)
print(f"Our MLE: mu_hat={mu_hat:.4f} (true {true_mu}), sigma_hat={np.sqrt(sigma2_hat):.4f} (true {true_sigma})")

scipy_mu, scipy_sigma = norm.fit(sample)  # independent cross-check via scipy's own MLE fitting
print(f"scipy.stats.norm.fit: mu={scipy_mu:.4f}, sigma={scipy_sigma:.4f}  (matches our closed form)")

# Fisher information for a Gaussian mean (sigma known) is I(mu) = 1/sigma^2 analytically.
# Confirm the Cramer-Rao/asymptotic-normality prediction Var(MLE) ~ 1/(n*I(mu)) = sigma^2/n
# by literally repeating the "collect n samples, compute the MLE" experiment thousands of times.
n_per_experiment, n_experiments = 40, 5000
mle_estimates = np.array([
    rng.normal(true_mu, true_sigma, size=n_per_experiment).mean() for _ in range(n_experiments)
])
empirical_var = mle_estimates.var()
fisher_info_mu = 1 / true_sigma ** 2
predicted_var = 1 / (n_per_experiment * fisher_info_mu)
print(f"\nEmpirical variance of the MLE across {n_experiments} repeated experiments (n={n_per_experiment} each): {empirical_var:.5f}")
print(f"Fisher-information-predicted variance sigma^2/n = 1/(n*I(mu)):                                {predicted_var:.5f}")

# EM from scratch: fit a 2-component 1D Gaussian mixture where cluster membership is a LATENT
# variable never directly observed -- only the mixed-together x values are seen.
rng2 = np.random.default_rng(2)
true_means, true_stds, true_weights = np.array([-2.0, 3.0]), np.array([1.0, 1.5]), np.array([0.4, 0.6])
component = rng2.choice(2, size=1000, p=true_weights)
data_gmm = rng2.normal(true_means[component], true_stds[component])

def fit_gmm_em(data, means, stds, weights, n_iters=50, reg_covar=1e-6, min_weight_mass=1e-8):
    """EM for a 2-component 1D Gaussian mixture, with two numerical safeguards this notebook's
    earlier version did not have:

    1) `Nk = responsibilities.sum(axis=1)` (the effective number of points "assigned" to each
       component) can shrink toward zero if a component's responsibility mass collapses -- the
       well-known GMM degeneracy where one component's std shrinks onto a single point (or a
       component gets outcompeted entirely by the other). `means = (responsibilities @ data) / Nk`
       and the `stds` update both divide by Nk directly: if Nk underflows to (numerically) zero,
       this is a silent 0/0 = nan, corrupting that component's parameters for every subsequent
       iteration. We floor Nk at `min_weight_mass` before dividing.
    2) `total_likelihood = comp_likelihoods.sum(axis=0)` can underflow to exactly 0.0 for any
       point far in the tails of BOTH components at once, making `responsibilities = comp_likelihoods
       / total_likelihood` another silent 0/0. We floor `total_likelihood` the same way.
    3) `stds` can (numerically) reach exactly 0 if a component collapses onto a single repeated
       value -- production code (e.g. sklearn.mixture.GaussianMixture's `reg_covar`) adds a small
       floor to the variance for exactly this reason; we do the same via `reg_covar`.
    """
    means, stds, weights = means.copy(), stds.copy(), weights.copy()
    log_likelihoods = []
    for _ in range(n_iters):
        # E-step: responsibility r_ik = P(z_i=k | x_i, theta_current) -- Bayes' theorem (03.05)
        # applied to "which component generated this point", using the CURRENT parameter guess
        comp_likelihoods = np.array([weights[k] * norm(means[k], stds[k]).pdf(data) for k in range(2)])
        total_likelihood = np.maximum(comp_likelihoods.sum(axis=0), 1e-300)   # guard against 0/0
        responsibilities = comp_likelihoods / total_likelihood                 # shape (2, n), columns sum to 1
        log_likelihoods.append(np.sum(np.log(total_likelihood)))               # observed-data log-likelihood

        # M-step: maximize the expected complete-data log-likelihood -- now a weighted Gaussian
        # MLE, closed-form because responsibilities (not raw counts) act as soft cluster memberships
        Nk = np.maximum(responsibilities.sum(axis=1), min_weight_mass)         # guard against Nk -> 0
        weights = Nk / len(data)
        means = (responsibilities @ data) / Nk
        variances = (responsibilities * (data - means[:, None]) ** 2).sum(axis=1) / Nk
        stds = np.sqrt(variances + reg_covar)                                   # guard against std -> 0
    return means, stds, weights, log_likelihoods

means0, stds0, weights0 = np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([0.5, 0.5])  # deliberately off
means, stds, weights, log_likelihoods = fit_gmm_em(data_gmm, means0, stds0, weights0, n_iters=50)

order = np.argsort(means)  # sort recovered components by mean to align with true_means (already sorted)
print(f"\nEM recovered means:   {np.round(means[order], 3)}  (true: {true_means})")
print(f"EM recovered stds:    {np.round(stds[order], 3)}  (true: {true_stds})")
print(f"EM recovered weights: {np.round(weights[order], 3)}  (true: {true_weights})")

diffs = np.diff(log_likelihoods)
print(f"\nLog-likelihood monotonically non-decreasing across all EM iterations? {bool(np.all(diffs >= -1e-8))}")
print(f"Log-likelihood: {log_likelihoods[0]:.2f} -> {log_likelihoods[-1]:.2f}  (EM's defining guarantee)")

# --- Regression test: a DEGENERATE initialization (both components start on top of each other,
# with a near-zero starting std) pushes one component's responsibility mass toward zero within
# a few iterations -- exactly the collapse the guards above exist to survive. ---
print("\n--- Regression test: EM from a degenerate, collapse-prone initialization ---")
means_bad, stds_bad, weights_bad = np.array([0.05, -0.05]), np.array([1e-4, 1e-4]), np.array([0.5, 0.5])
means_r, stds_r, weights_r, ll_r = fit_gmm_em(data_gmm, means_bad, stds_bad, weights_bad, n_iters=50)
print(f"Recovered means:   {np.round(means_r, 4)}")
print(f"Recovered stds:    {np.round(stds_r, 4)}")
print(f"Recovered weights: {np.round(weights_r, 4)}")
print(f"All parameters finite (no nan from a collapsed component): "
      f"{np.all(np.isfinite(means_r)) and np.all(np.isfinite(stds_r)) and np.all(np.isfinite(weights_r))}")
assert np.all(np.isfinite(means_r)) and np.all(np.isfinite(stds_r)) and np.all(np.isfinite(weights_r)), \
    "EM must stay finite even from a collapse-prone initialization"

## 🤖 Why This Matters for AI

Training a neural network with cross-entropy loss **is** maximum likelihood estimation:
cross-entropy is, by direct algebraic identity, the negative log-likelihood of the labels under
the model's predicted distribution, averaged over the batch. This isn't a loose analogy — the
cell below computes both quantities, at every training step, on an actual model trained with a
real optimizer, and shows they are the same number up to the sign flip and the $1/n$
normalization the Theory section derived.

---

In [ ]:
# THE central claim of this notebook, shown numerically: training a logistic-regression
# classifier by minimizing cross-entropy loss IS maximizing likelihood -- not similar, IDENTICAL.
rng = np.random.default_rng(0)
torch.manual_seed(0)

n_samples, n_features = 300, 4
X_np = rng.normal(size=(n_samples, n_features))
true_w = np.array([1.5, -2.0, 0.5, 1.0])
probs_true = 1 / (1 + np.exp(-(X_np @ true_w)))
y_np = (rng.uniform(size=n_samples) < probs_true).astype(np.float32)

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

model = nn.Linear(n_features, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
bce_loss_fn = nn.BCEWithLogitsLoss(reduction="mean")  # THE cross-entropy loss, for 2 classes

ce_losses, neg_ll_over_n = [], []
print(f"{'step':>5} | {'cross-entropy loss':>19} | {'log-likelihood':>15} | {'accuracy':>8}")
for step in range(151):
    logits = model(X).squeeze(-1)
    loss = bce_loss_fn(logits, y)  # what the optimizer actually minimizes

    with torch.no_grad():
        probs = torch.sigmoid(logits)
        # the SAME quantity computed independently, straight from this notebook's own
        # log-likelihood definition ell(theta) = sum_i log p(y_i | x_i; theta)
        log_likelihood = torch.sum(y * torch.log(probs) + (1 - y) * torch.log(1 - probs)).item()
        accuracy = ((probs > 0.5).float() == y).float().mean().item()
    ce_losses.append(loss.item())
    neg_ll_over_n.append(-log_likelihood / n_samples)

    if step % 30 == 0:
        print(f"{step:5d} | {loss.item():19.6f} | {log_likelihood:15.4f} | {accuracy:8.3f}")

    optimizer.zero_grad()
    loss.backward()   # the TRUE gradient of the cross-entropy loss -- no heuristic update rule
    optimizer.step()

max_diff = np.max(np.abs(np.array(ce_losses) - np.array(neg_ll_over_n)))
print(f"\nmax |cross-entropy loss - (-log-likelihood / n)| across all {len(ce_losses)} steps: {max_diff:.2e}")
print("Zero up to floating point -- minimizing cross-entropy loss and maximizing the log-likelihood")
print("are the exact same optimization problem, confirmed at every single step of real training.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a Bernoulli($p$) likelihood with $k=7$ successes out of $n=20$ trials, derive
   $\hat{p}_{\text{MLE}}$ by setting $\frac{d\ell}{dp}=0$ by hand, where
   $\ell(p) = k\log p + (n-k)\log(1-p)$. As a sanity check, confirm your answer lies in $[0,1]$
   (a probability estimate outside that range would signal an error), then verify it against a
   grid search over `p_grid = np.linspace(0.01, 0.99, 981)` that picks the `p` maximizing
   `k*np.log(p) + (n-k)*np.log(1-p)`.

<details>
<summary>💡 Solution</summary>

$\frac{d\ell}{dp} = \frac{k}{p} - \frac{n-k}{1-p}$. Setting this to zero:
$\frac{k}{p}=\frac{n-k}{1-p} \implies k(1-p) = (n-k)p \implies k - kp = np - kp \implies k = np
\implies \hat p_{\text{MLE}} = k/n$. For $k=7,n=20$: $\hat p_{\text{MLE}} = 7/20 = 0.35$, which
is indeed in $[0,1]$. A grid search over `p_grid` finds its maximum log-likelihood exactly at
`p=0.35` (up to the grid's resolution), matching the closed form.

</details>

### 💭 UNDERSTAND
2. The notebook's `gaussian_mle` computes `sigma2_hat = np.mean((data - mu_hat) ** 2)` — dividing
   by $n$, not $n-1$. It's a mathematical fact that $\mathbb{E}[\hat\sigma^2_{\text{MLE}}] =
   \frac{n-1}{n}\sigma^2_{\text{true}}$ — the MLE variance estimator is **biased**, systematically
   *underestimating* the true variance on average, especially for small $n$. Explain in your own
   words why using $\hat\mu = \bar x$ (itself estimated from the same data) instead of the true
   unknown $\mu$ inside $\sum_i(x_i-\hat\mu)^2$ is exactly what causes this underestimation.

<details>
<summary>💡 Solution</summary>

$\bar x$ is computed to minimize $\sum_i (x_i - c)^2$ over all possible constants $c$ (a basic
least-squares fact — the sample mean is the value that makes the sum of squared deviations as
small as possible for *this particular sample*). Substituting the *true* unknown $\mu$ into that
same sum would generally make it slightly larger, since $\mu$ was not chosen to minimize this
specific sample's deviations. So $\sum_i(x_i-\bar x)^2 \le \sum_i(x_i-\mu)^2$ for every sample,
which means the "plug in $\bar x$" version systematically comes out smaller — and averaged over
many samples, `sigma2_hat` (built from $\bar x$) undershoots the true variance, which is why
dividing by the smaller $n-1$ instead of $n$ exactly corrects the bias (this is Bessel's
correction, and also why `03.03`-style variance formulas in statistics packages often default to
$n-1$).

</details>

### ✏️ DERIVE
3. Derive the bias of the MLE variance estimator stated in exercise 2:
   $\mathbb{E}[\hat\sigma^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2$. Start from the algebraic
   identity $\sum_i(x_i-\bar x)^2 = \sum_i(x_i-\mu)^2 - n(\bar x-\mu)^2$ (expand both sides to
   confirm it, or take it as given), then take expectations of both sides using
   $\mathbb{E}[(x_i-\mu)^2]=\sigma^2$ and $\text{Var}(\bar x)=\sigma^2/n$.

<details>
<summary>💡 Solution outline</summary>

Take $\mathbb{E}[\cdot]$ of both sides of $\sum_i(x_i-\bar x)^2 = \sum_i(x_i-\mu)^2 -
n(\bar x-\mu)^2$. For the first term on the right: $\mathbb{E}\big[\sum_i(x_i-\mu)^2\big] =
\sum_i\mathbb{E}[(x_i-\mu)^2] = n\sigma^2$ (each term is exactly the true variance, by
definition). For the second: $\mathbb{E}[n(\bar x-\mu)^2] = n\,\mathbb{E}[(\bar x-\mu)^2] =
n\,\text{Var}(\bar x) = n\cdot\frac{\sigma^2}{n} = \sigma^2$ (since $\bar x$ is unbiased for
$\mu$, $\mathbb{E}[(\bar x-\mu)^2]$ is exactly $\text{Var}(\bar x)$). Combining:
$$\mathbb{E}\Big[\sum_i(x_i-\bar x)^2\Big] = n\sigma^2 - \sigma^2 = (n-1)\sigma^2.$$
Since $\hat\sigma^2_{\text{MLE}} = \frac{1}{n}\sum_i(x_i-\bar x)^2$, dividing both sides by $n$:
$$\mathbb{E}[\hat\sigma^2_{\text{MLE}}] = \frac{(n-1)\sigma^2}{n} = \frac{n-1}{n}\sigma^2,$$
confirming the claim — and confirming that $\frac{1}{n-1}\sum_i(x_i-\bar x)^2$ (dividing by
$n-1$ instead) is exactly unbiased, since it rescales this result back up to $\sigma^2$ exactly.

</details>

### 🐍 IMPLEMENT
4. In the Fisher information cell, repeat the "collect $n$ samples, compute the MLE, repeat many
   times" experiment for `n_per_experiment` in `[10, 40, 160, 640]`, recording the empirical
   variance of the resulting MLE estimates at each `n`. Compare against the Fisher-information
   prediction $\sigma^2/n$ at each value, and fit a line to `log(empirical_variance)` vs.
   `log(n)`.

<details>
<summary>✅ How to know you're right</summary>

At each of the four sample sizes, the empirical variance (across, say, 5000 repeated
experiments) should land within roughly 5-10% of the predicted $\sigma^2/n$ value — e.g. at
`n=40` with `true_sigma=2.0`, the prediction is exactly `0.1`, and a real run should land close
to that. The fitted slope of `log(empirical_variance)` against `log(n)` should come out close to
$-1$ (representative runs give slopes around $-0.98$ to $-1.0$), confirming the predicted
$1/n$ scaling — a slope far from $-1$ (e.g. close to $0$ or steeper than $-2$) would indicate a
bug in the experiment loop rather than a real deviation from theory, since this scaling is exact
for the Gaussian mean's MLE.

</details>

### 🤖 APPLY
5. Requests hitting an ML inference server arrive with inter-arrival times that are well-modeled
   as Exponential($\lambda$) — the same distribution `03.06` introduced as a Poisson-process
   waiting time. Given a logged sample of observed inter-arrival times (simulate one with
   `rng.exponential(scale=1/true_rate, size=n)` for some `true_rate` and `n` of your choosing),
   derive and implement `exponential_rate_mle(data)` returning $\hat\lambda_{\text{MLE}} =
   1/\bar x$ (derive this the same way exercise 1 derived the Bernoulli MLE: write
   $\ell(\lambda)=n\log\lambda-\lambda\sum_i x_i$, set $\frac{d\ell}{d\lambda}=0$, solve), and
   use it to estimate the server's request rate $\lambda$ (requests per unit time).

<details>
<summary>✅ What you should observe</summary>

Setting $\frac{d\ell}{d\lambda} = \frac{n}{\lambda}-\sum_i x_i = 0$ gives $\hat\lambda_{\text{MLE}}
= n/\sum_i x_i = 1/\bar x$. Implemented and applied to simulated data (e.g. `true_rate=0.5`,
`n=2000`), `exponential_rate_mle(data)` should return a value close to the true rate (a
representative run with `true_rate=0.5` gives an estimate around `0.494`), converging closer to
the true rate as `n` grows, consistent with MLE consistency. This is the same closed-form-MLE
pattern as `gaussian_mle` in the notebook's own Implementation section, just applied to a
different distribution and a genuinely different AI-systems scenario (estimating a service's
request rate from logs, rather than fitting a model's own parameters to training data).

</details>

### 🚀 CHALLENGE
6. EM only guarantees the observed log-likelihood never *decreases* across iterations — it does
   **not** guarantee convergence to the global maximum. Investigate this directly with the
   notebook's own `fit_gmm_em` and `data_gmm`: try several deliberately poor initializations
   (e.g. both starting means far in one tail of the data with a tiny starting std for one of
   them, or both starting means placed close together on the same side of the data) alongside
   the notebook's own reasonable initialization. For each, confirm log-likelihood is still
   monotonically non-decreasing across iterations, but record and compare the *final* converged
   log-likelihood. Characterize what a "bad" converged solution looks like in terms of the
   recovered means/stds/weights (hint: watch for a component's std collapsing near zero).

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds at least one poor initialization (e.g. one starting mean placed far into
a sparse tail of `data_gmm` with a small starting std) that converges — monotonically, exactly
as EM guarantees — to a final log-likelihood noticeably worse than the well-initialized run's
(a representative comparison: around $-2427$ or $-2424$ for degenerate initializations vs.
around $-2254$ for a good one, a real and substantial gap, not a rounding difference), while
`np.diff(log_likelihoods)` still stays non-negative throughout every one of these runs. The
degenerate solutions typically show one component's weight collapsing toward a tiny fraction
(e.g. under $1\%$) of the data and its std shrinking toward the `reg_covar` floor, effectively
fitting a spike to one or two outlier-ish points while the other component absorbs everything
else — the classic GMM "variance collapse" local optimum. A strong answer explicitly connects
this back to the Theory section's careful wording ("never decreases" rather than "converges to
the maximum") and notes the standard practical mitigations: running EM from several random
initializations and keeping the best final log-likelihood, or using a smarter initialization
scheme (e.g. k-means++-style seeding) rather than arbitrary starting values.

</details>

---

## 📚 Further Reading
- Casella & Berger, *Statistical Inference*, Ch. 7 (Point Estimation) and Ch. 10 (Asymptotic Evaluations)
- Dempster, Laird & Rubin, ["Maximum Likelihood from Incomplete Data via the EM Algorithm"](https://www.jstor.org/stable/2984875) (1977)
- Goodfellow, Bengio & Courville, *Deep Learning*, Ch. 5.5 (Maximum Likelihood Estimation)

---

*Next notebook: [Hypothesis Testing and Confidence Intervals](08_hypothesis_testing_confidence.ipynb)*